In [ ]:
# Install nirdizati-light package
!pip install git+https://github.com/rgraziosi-fbk/nirdizati-light
# If asked to reload runtime to update numpy version, click "yes"

# Reinstall cupy-cuda12 to ensure compatibility
!pip uninstall -y cupy-cuda12x
!pip install cupy-cuda12x|

from google.colab import drive
drive.mount('/content/drive')

import sys
#sys.path.insert(0, '/content/drive/My Drive/Colab Notebooks/nirdizati-light_Test')
sys.path.insert(0, '/home/scala/.local/lib/python3.10/site-packages/')
sys.path.insert(0, '/home/scala/.local/bin')
#!source /home/scala/.virtualenvs/nirdizati-light/bin/activate

In [ ]:
# Download an example log
!mkdir datasets
!wget "https://drive.google.com/uc?export=download&id=1qcx8F7nFo20kENuvBKWfQLidgi54adlv" -O './datasets/bpic2012_O_ACCEPTED-COMPLETE_trunc.xes'

import sys
#print(sys.prefix)
#print(sys.base_prefix)
#!pip list

import os
print(os.environ["PATH"])
import numpy as np
print("My numpy version is: ", np.__version__)
import pandas as pd
print("My numpy version is: ", pd.__version__)
import numba
print("My numba version is: ", numba.__version__)
from numba import cuda
print(cuda.gpus)

# Prepare environment

In [13]:
import random
import pm4py
from nirdizati_light.log.common import get_log, split_train_val_test
from nirdizati_light.encoding.common import get_encoded_df, EncodingType
from nirdizati_light.encoding.constants import TaskGenerationType, PrefixLengthStrategy, EncodingTypeAttribute
from nirdizati_light.encoding.time_encoding import TimeEncodingType
from nirdizati_light.labeling.common import LabelTypes

In [14]:
from enum import Enum

from pm4py.objects.log.obj import EventLog, Trace

class EncodingType(Enum):
	SIMPLE = 'simple'
	FREQUENCY = 'frequency'
	COMPLEX = 'complex'
	DECLARE = 'declare'
	LORELEY = 'loreley'
	LORELEY_COMPLEX = 'loreley_complex'

class EncodingTypeAttribute(Enum):
	LABEL = 'label'
	ONEHOT = 'onehot'

class TaskGenerationType(Enum):
	ONLY_THIS = 'only_this'
	ALL_IN_ONE = 'all_in_one'

class PrefixLengthStrategy(Enum):
	FIXED = 'fixed'
	PERCENTAGE = 'percentage'
	TARGET_EVENT = 'target_event'

def get_prefix_length(trace: Trace, prefix_length: float, prefix_length_strategy, target_event=None) -> int:
	if prefix_length_strategy == PrefixLengthStrategy.FIXED.value:
		return int(prefix_length)
	elif prefix_length_strategy == PrefixLengthStrategy.PERCENTAGE.value:
		return int(prefix_length * len(trace))
	elif prefix_length_strategy == PrefixLengthStrategy.TARGET_EVENT.value:
		if target_event is not None:
			try:
				index = [e['concept:name'] for e in trace].index(target_event) + 1
			except ValueError:
				return 0
			return index
		else:
			return 0
	else:
		raise Exception('Wrong prefix_length strategy')

def get_max_prefix_length(log: EventLog, prefix_length: float, prefix_length_strategy, target_event) -> int:
	prefix_lengths = [get_prefix_length(trace, prefix_length, prefix_length_strategy, target_event) for trace in log]
	return max(prefix_lengths)

## Chosen encoding for categorical features: OneHot

In [15]:
CONF = {
    # path to log
    'data': 'BPIC11_f1.csv',
    # train-validation-test set split percentages
    'train_val_test_split': [0.7, 0.1, 0.2],

    # path to output folder
    'output': 'output_data',

    'prefix_length_strategy': PrefixLengthStrategy.FIXED.value,
    'prefix_length': 15,

    # whether to use padding or not in encoding
    'padding': True,
    # which encoding to use
    'feature_selection': EncodingType.COMPLEX.value,
    # which attribute encoding to use
    # Forse vogliamo OneHot
    'attribute_encoding': EncodingTypeAttribute.ONEHOT.value,
    # which time encoding to use
    'time_encoding': TimeEncodingType.NONE.value,

    # the label to be predicted (e.g. outcome, next activity)
    'labeling_type': LabelTypes.ATTRIBUTE_STRING.value,
    # whether the model should be trained on the specified prefix length (ONLY_THIS) or to every prefix in range [1, prefix_length] (ALL_IN_ONE)
    'task_generation_type': TaskGenerationType.ALL_IN_ONE.value,
    'target_event': None,
    'seed': 49,
}

In [17]:
log = get_log(filepath=CONF['data'], separator=';')

log[0][0].keys()

In [18]:
encoder, full_df = get_encoded_df(
  log=log,
  feature_encoding_type=CONF['feature_selection'],
  prefix_length=CONF['prefix_length'],
  prefix_length_strategy=CONF['prefix_length_strategy'],
  time_encoding_type=CONF['time_encoding'],
  attribute_encoding=CONF['attribute_encoding'],
  padding=CONF['padding'],
  labeling_type=CONF['labeling_type'],
  task_generation_type=CONF['task_generation_type'],
  target_event=CONF['target_event'],
)

/home/scala/projectsLuigi/nirdizati-light/nirdizati_light/encoding/time_encoding.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_output[column_name] = current_time
/home/scala/projectsLuigi/nirdizati-light/nirdizati_light/encoding/time_encoding.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_output[column_name] = current_time
/home/scala/projectsLuigi/nirdizati-light/nirdizati_light/encoding/time_encoding.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse

column: Age_1 considered number, top 5 values are: [33, 33, 33, 33, 33]
column: Number of executions_1 considered number, top 5 values are: [1, 1, 1, 1, 1]
column: event_nr_1 considered number, top 5 values are: [1, 1, 1, 1, 1]
column: hour_1 considered number, top 5 values are: [23, 23, 23, 23, 23]
column: month_1 considered number, top 5 values are: [1, 1, 1, 1, 1]
column: open_cases_1 considered number, top 5 values are: [5, 5, 5, 5, 5]
column: time:timestamp_1 considered number, top 5 values are: [1104706800.0, 1104706800.0, 1104706800.0, 1104706800.0, 1104706800.0]
column: timesincecasestart_1 considered number, top 5 values are: [0.0, 0.0, 0.0, 0.0, 0.0]
column: timesincelastevent_1 considered number, top 5 values are: [0.0, 0.0, 0.0, 0.0, 0.0]
column: timesincemidnight_1 considered number, top 5 values are: [1380, 1380, 1380, 1380, 1380]
column: weekday_1 considered number, top 5 values are: [6, 6, 6, 6, 6]
column: Age_2 considered number, top 5 values are: [0, 33, 33, 33, 33]
c

In [19]:
full_df = full_df.copy()
#encoder.decode(full_df)
#encoder.get_values(full_df)

label_columns = [item for item in full_df.columns if 'label' in item ]
label_columns

def check_att(att, prefix_length):
     try:
          return not ('prefix_' in att or int( att[ len(att) - att[::-1].index('_') : ] ) in range(1,prefix_length+1) )
     except ValueError:
          return True

def is_trace_attribute(att):
     try:
        cur_length = int( att[ len(att) - att[::-1].index('_') : ] )
        #print(att, ': ', att, 'cur_length: ', cur_length, ' : ', False)
        return False #not ('prefix_' in att or 'Prefix_' in att) #or not cur_length in range(1,prefix_length+1)
     except ValueError:
        #print(att, ': ValueError: True' )
        return True

## Nuova versione del metodo get_tensor

Pietro: Funziona solo se prefix length è uguale a quello nel preprocess (???)


In [69]:
import numpy as np
import torch
from pandas import DataFrame
from funcy import flatten

def get_tensor_alt(df, prefix_length, aggregate = True, trace_att_filter = ['trace_id', 'label'], event_att_filter = ['label', 'prefix', 'Prefix']):
    
    def is_trace_attribute(att):
     try:
        return not int( att[ len(att) - att[::-1].index('_') : ] ) > 0
     except ValueError:
        return True        
        
    def create_numerical_feature_list(trace, selected_attributes):
        feature_count = 0
        numerical_feature_list = []
        #print('trace: ', trace)
        for feat_name, feat_values in trace.items():
            if feat_name in selected_attributes:
                #print('\nisinstance(feat_values, tuple): ', isinstance(feat_values, tuple))
                if isinstance(feat_values, tuple):
                    #print('not inserted feature: ', feat_name, '\n \t with feat_values', feat_values)
                    feature_count += len(feat_values)
                else:
                    numerical_feature_list.append(feature_count)
                    #print('inserting feature: ', feat_name, '\n \t with count ', feature_count)
                    feature_count += 1
        return numerical_feature_list
        
    trace_attributes = [ att for att in df.columns if is_trace_attribute(att) ]
    event_attributes = [ att[:-2] for att in df.columns if att[-2:] == '_1' and not is_trace_attribute(att)]
    #print('trace_attributes: ', trace_attributes)
    #print('event_attributes: ', event_attributes)

    reshaped_data = {
        trace_index: {
            prefix_index:
                list(flatten(
                    feat_values if isinstance(feat_values, tuple) else [feat_values]
                    for feat_name, feat_values in trace.items()
                    if feat_name in [trace_attribute for trace_attribute in trace_attributes if trace_attribute not in trace_att_filter]  + [event_attribute + '_' + str(prefix_index) for event_attribute in event_attributes if event_attribute not in event_att_filter]
                ))
            for prefix_index in range(1, prefix_length + 1)
        }
        for trace_index, trace in df.iterrows()
    }

    flattened_feature_length = max(
        len(reshaped_data[trace][prefix])
        for trace in reshaped_data
        for prefix in reshaped_data[trace]
    )

    trace_number = len(df)
    tensor = torch.zeros((
        trace_number,               # samples
        prefix_length,              # time steps
        flattened_feature_length    # features per single time step (trace and event attributes)
    ))

    selected_attributes = [trace_attribute for trace_attribute in trace_attributes if trace_attribute not in trace_att_filter]  + [event_attribute + '_1' for event_attribute in event_attributes if event_attribute not in event_att_filter]

    numerical_feature_list = create_numerical_feature_list(df.loc[0], selected_attributes) if aggregate else []
    #print('numerical_feature_list: ', numerical_feature_list)
    
    for i, trace_index in enumerate(reshaped_data):  # prefix
        for j, prefix_index in enumerate(reshaped_data[trace_index]):  # steps of the prefix
            for single_flattened_value in range(len(reshaped_data[trace_index][prefix_index])):
                tensor[i, j, single_flattened_value] = reshaped_data[trace_index][prefix_index][single_flattened_value]
                if aggregate and j in numerical_feature_list:
                    tensor[i, j, single_flattened_value] /= trace_number
                    
    #print('tensor.shape: ', tensor.shape)
    if aggregate:        
        return torch.sum(tensor, dim=1)
    else: 
        return tensor                    

In [24]:
df = full_df.iloc[0:120]
#df = full_df.head(120)
df 

,trace_id,prefix_1,Age_1,Diagnosis_1,Diagnosis Treatment Combination ID_1,Diagnosis code_1,Number of executions_1,Producer code_1,Section_1,Specialism code_1,...,hour_15,label_15,month_15,open_cases_15,time:timestamp_15,timesincecasestart_15,timesincelastevent_15,timesincemidnight_15,weekday_15,label
0,0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.175,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,1
1,0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.175,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,1
2,0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.175,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,1
3,0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.175,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,1
4,0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.175,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,7,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",0.350,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,0
116,7,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",0.350,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,0
117,7,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",0.350,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 1.0, 0.0)",...,0.0,"(1.0, 0.0, 0.0)",0.000000,0.000000,"(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,0.0,0.0,0.000000,0
118,7,"(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...",0.350,"(0.0, 0.0, 0.0, 0.0, 0.0, 0

In [70]:
test_tensor = get_tensor_alt(df, 4)
test_tensor
print('tensor shape: ', test_tensor.shape)


tensor shape:  torch.Size([120, 1566])


## Encoding

The function `get_encoded_df` encodes data from a process log to create:
1. An **encoder object** (`encoder`) that processes features or labels.
2. A fully encoded **DataFrame** (`full_df`) for machine learning tasks.

### Parameters:

1. **`log`**:
   - Likely a process event log or dataset containing sequential events for one or more processes.
   - Columns might include `case_id`, `activity`, `timestamp`, and other attributes.

2. **`feature_encoding_type`**:
   - Specifies the type of feature encoding:
     - `frequency`: Frequency-based encoding (e.g., count of activities).
     - `ordinal`: Maps categorical features to integers.
     - `one_hot`: One-hot encoding for categorical features.
     - `embedding`: Use embeddings for numerical representation.

3. **`prefix_length`**:
   - Determines the length of process prefixes to be used.
   - Affects how much history (number of events) is considered for features.

4. **`prefix_length_strategy`**:
   - Defines how prefixes are handled:
     - `fixed`: Use a fixed number of steps (e.g., only first $N$ steps).
     - `dynamic`: Variable-length prefixes, potentially based on case characteristics.

5. **`time_encoding_type`**:
   - Encodes timestamp-related features (e.g., weekday, hour, month, time since last event, time since case start):
     - `NONE`: Does not encode the relative time attributes.
     - `DATE`: Extracts date-related features like day-of-week or month.
     - `DURATION`: Encodes time differences between events, time since case start, etc.
     - `DATE_AND_DURATION`: Combines date and duration encodings.

6. **`attribute_encoding`**:
   - Specifies how to encode additional attributes:
     - `LABEL`: Encode attributes as labels (e.g., 'A', 'B', 'C').
     - `ONEHOT`: Use one-hot encoding for attributes.

7. **`padding`**:
   - Ensures uniform input length by padding shorter prefixes with default values (e.g., zeros).
   - Useful for models like RNNs or LSTMs requiring uniform input dimensions.

8. **`labeling_type`**:
   - Defines how labels are generated for classification or prediction:
     - `NEXT_ACTIVITY`: Predict the next activity in the process.
     - `ATTRIBUTE_STRING`: Predict the final outcome of a process instance.
     - `ATTRIBUTE_NUMBER`: Predict a numerical attribute value (e.g., case duration).
     - `REMAINING_TIME`: Predict the remaining time until case completion.
     - `DURATION`: Predict the duration of a case.

9. **`task_generation_type`**:
   - Specifies the type of task:
     - `CLASSIFICATION`: Assign discrete classes (e.g., case outcome as 'successful' or 'failed').
     - `REGRESSION`: Predict continuous values (e.g., REMAINING_TIME, DURATION).

10. **`target_event`**:
    - Focuses on a specific event in the process for next activity predictive tasks.
    - Example: A particular milestone activity or timestamp.
        - 
### Outputs:

1. **`encoder`**:
   - Object or function that transforms raw features into encoded forms.
   - Stores mappings (e.g., one-hot encoding maps, label encoders).

2. **`full_df`**:
   - Fully encoded DataFrame for model training.
   - Includes encoded features, labels, and optional padding for consistency.

### <span style="color: red;">WARNING:</span> When using encoder.decode(full_df), this modifies the full_df DataFrame in place. To encode the dataframe again, you need to re-run the encoding process with encoder.encode(full_df).